## Function for calculating sliding window MSDs

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

def calculate_sliding_tamsd_with_slope(single_traj: pd.DataFrame, window_size: int = 5, step_size: int = 1):
    """Calculates time-averaged MSDs of one track in a given ROI using sliding windows and also calculates the slope of each MSD curve.
    
    Inputs: 
        single_traj: DataFrame containing coordinates of one trajectory in an ROI, sorted by frame.
        window_size: Size of the sliding window.
        step_size: Step size for sliding the window.
        
    Return:
        results: DataFrame containing window index, lags, time-averaged MSDs, and slope of each MSD curve for each window.
    """
    tvalues = single_traj["frame"].values
    tvalues = tvalues[:, None] - tvalues
    
    window_index = []
    final_lags = []
    tamsd = []
    
    # Loop over sliding windows
    for i in range(0, len(single_traj) - window_size + 1, step_size):
        window_traj = single_traj.iloc[i:i+window_size]  # Get data for the current window
        
        # Calculate pair-wise differences within the window
        tvalues_window = window_traj["frame"].values
        tvalues_window = tvalues_window[:, None] - tvalues_window
        
        for lag in range(1, len(window_traj) + 1):
            x, y = np.where(tvalues_window == lag)
            
            if len(x) < 2:  # Skip if less than 2 points in the window for given lag
                continue
            
            tmp_tamsd = np.mean(
                np.square(
                    window_traj.iloc[x][["x", "y"]].values
                    - window_traj.iloc[y][["x", "y"]].values
                )
            )
            
            final_lags.append(lag)
            tamsd.append(tmp_tamsd)
            window_index.append(i)
    
    # Create DataFrame with time-averaged MSDs
    df = pd.DataFrame({"window_index": window_index, "lags": final_lags, "tamsd": tamsd})
    
    # Calculate the slope for each window using linear regression
    slopes = []
    for i, window in df.groupby("window_index"):
        model = LinearRegression()
        x_values = window["lags"].values.reshape(-1, 1)
        y_values = window["tamsd"].values
        model.fit(x_values, y_values)
        slopes.append(model.coef_[0])
    
    # Add slope column to the DataFrame
    df["slope"] = np.repeat(slopes, df.groupby("window_index").size())
    
    return df
